In [5]:
import os
import json
import torch
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel
# ----------------------------
# 配置
# ----------------------------
SAVE_DIR = "unsafe_embeddings"
CLIP_MODEL_Tokennider = "/root/autodl-tmp/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/tokenizer"
CLIP_MODEL_Text_encoder="/root/autodl-tmp/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = CLIPTokenizer.from_pretrained(CLIP_MODEL_Tokennider)
text_encoder = CLIPTextModel.from_pretrained(CLIP_MODEL_Text_encoder).to(DEVICE)
text_encoder.eval()
os.makedirs(SAVE_DIR, exist_ok=True)
@torch.no_grad()
def encode_prompts(prompts, batch_size=256):
    embeddings = []

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i + batch_size]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = text_encoder(**tokens)
        # CLIP 使用 last_hidden_state 的 [EOS] token
        emb = outputs.last_hidden_state[:, -1, :]  # [B, D]
        emb = emb / emb.norm(dim=-1, keepdim=True)  # normalize

        embeddings.append(emb.cpu())

    return torch.cat(embeddings, dim=0)
def build_from_txt(txt_path, save_dir=SAVE_DIR, batch_size=256):
    """Read sentences from a .txt file, encode them with `encode_prompts`,
    and save the resulting matrix to `save_dir` as <basename>.pt.

    Returns the path to the saved .pt file.
    """
    if not os.path.exists(txt_path):
        raise FileNotFoundError(f"Input file not found: {txt_path}")
    prompts=[]
    with open(txt_path, "r", encoding="utf-8") as f:
        # Keep non-empty, stripped lines as prompts
        if txt_path.endswith(".json"):
            lines=json.load(f)
            for line in lines:
                prompts.append(line["prompt"])
        else:
            prompts = [line.strip() for line in f if line.strip()]

    if len(prompts) == 0:
        raise ValueError(f"No non-empty lines found in {txt_path}")

    emb = encode_prompts(prompts, batch_size=batch_size)

    base = os.path.splitext(os.path.basename(txt_path))[0]
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{base}.pt")

    torch.save(
        {
            "label": base,
            "embeddings": emb,        # [N, D]
            "num_prompts": len(prompts),
            "dim": emb.shape[1]
        },
        save_path
    )

    return save_path

In [7]:
#build_from_txt("/root/autodl-tmp/concept_detection/generated/bloody_100.json")
build_from_txt("/root/autodl-tmp/concept_detection/generated/nude_100.json")
build_from_txt("/root/autodl-tmp/concept_detection/generated/violence_100.json")

100%|██████████| 1/1 [00:17<00:00, 17.08s/it]


'unsafe_embeddings/violence_100.pt'